In [21]:
import sys
import csv
import os
import regex as re
import numpy as np

# /workspaces/collaborative-scene-graphs/metrics/2025-02-19T14:35-OG-completed
# /workspaces/collaborative-scene-graphs/metrics/2025-02-19T19:50-MaskCLIP-completed
# /workspaces/collaborative-scene-graphs/metrics/2025-02-19T23:58
eval_sessions = """
/workspaces/collaborative-scene-graphs/metrics/2025-02-26T19:47-final1
/workspaces/collaborative-scene-graphs/metrics/2025-02-27T11:30-final2
/workspaces/collaborative-scene-graphs/metrics/2025-02-28T00:16-final3
""".strip().split(
    "\n"
)

use_se2 = True

n_sessions = len(eval_sessions)

# dim1: (1,2,3) agents, dim2: (all,sel) gt, dim3: (p,r,e_all,e_assoc)
metrics_by_n_agents_by_gt_by_type = np.zeros((3, 2, 4))
counts_by_n_agents_by_gt_by_type = np.zeros((3, 2, 4))

for session in eval_sessions:
    runs = os.listdir(session)
    for run in runs:
        match = re.match(r"agents-(\d+)_remove_dyn_objs-(True|False)", run)
        if match is None:
            print(f"skipping {run}")
            continue
        agents, rdo = match.groups()
        if rdo == "True":
            continue

        n_agents = len(agents)
        d = os.path.join(session, run)
        ate_fname = "intersections.csv" if use_se2 else "ate.csv"
        with open(d + "/" + ate_fname, "r") as f:
            reader = csv.DictReader(f)
            these_metrics = list(reader)[-1]
            for i, gt in enumerate(["all", "sel"]):
                for j, metric in enumerate([f"precision_{gt}", f"recall_{gt}", f"mean_dst_{gt}_all", f"mean_dst_{gt}_assoc"]):
                    metric_val = float(these_metrics[metric])
                    metrics_by_n_agents_by_gt_by_type[n_agents - 1, i, j] += metric_val
                    counts_by_n_agents_by_gt_by_type[n_agents - 1, i, j] += 1

metrics_by_n_agents_by_gt_by_type /= counts_by_n_agents_by_gt_by_type
print("precision, recall, mean_dst_all, mean_dst_sel")
print("all_gt")
print(metrics_by_n_agents_by_gt_by_type[:, 0, :])
print("sel_gt")
print(metrics_by_n_agents_by_gt_by_type[:, 1, :])

precision, recall, mean_dst_all, mean_dst_sel
all_gt
[[ 0.85515873  0.50724638 25.6035383  16.53367668]
 [ 0.83935414  0.5289855  26.6763916  15.72593742]
 [ 0.83333331  0.54347825 26.72062556 14.44442081]]
sel_gt
[[ 0.76746031  0.91304348 47.46117062 16.24825732]
 [ 0.77033387  0.9710145  46.50004154 15.5751933 ]
 [ 0.76666665  1.         45.67255529 14.47678598]]


In [22]:
table = [
    # turned
    "\\multirow{1}{*}{\\textbf{1 Agent}}  & & \\cmark & " + "{:.4f} & {:.4f} & {:.4f} & {:.4f} \\\\".format( metrics_by_n_agents_by_gt_by_type[0, 1, 0], metrics_by_n_agents_by_gt_by_type[0, 1, 1], metrics_by_n_agents_by_gt_by_type[0, 1, 2], metrics_by_n_agents_by_gt_by_type[0, 1, 3],),
    "\\multirow{1}{*}{\\textbf{2 Agents}} & & \\cmark & " + "{:.4f} & {:.4f} & {:.4f} & {:.4f} \\\\".format( metrics_by_n_agents_by_gt_by_type[1, 1, 0], metrics_by_n_agents_by_gt_by_type[1, 1, 1], metrics_by_n_agents_by_gt_by_type[1, 1, 2], metrics_by_n_agents_by_gt_by_type[1, 1, 3],),
    "\\multirow{1}{*}{\\textbf{3 Agents}} & & \\cmark & " + "{:.4f} & {:.4f} & {:.4f} & {:.4f} \\\\".format( metrics_by_n_agents_by_gt_by_type[2, 1, 0], metrics_by_n_agents_by_gt_by_type[2, 1, 1], metrics_by_n_agents_by_gt_by_type[2, 1, 2], metrics_by_n_agents_by_gt_by_type[2, 1, 3],),
    "\\midrule",
    # all
    "\\multirow{1}{*}{\\textbf{1 Agent}}  & \\cmark & & " + "{:.4f} & {:.4f} & {:.4f} & {:.4f} \\\\".format( metrics_by_n_agents_by_gt_by_type[0, 0, 0], metrics_by_n_agents_by_gt_by_type[0, 0, 1], metrics_by_n_agents_by_gt_by_type[0, 0, 2], metrics_by_n_agents_by_gt_by_type[0, 0, 3],),
    "\\multirow{1}{*}{\\textbf{2 Agents}} & \\cmark & & " + "{:.4f} & {:.4f} & {:.4f} & {:.4f} \\\\".format( metrics_by_n_agents_by_gt_by_type[1, 0, 0], metrics_by_n_agents_by_gt_by_type[1, 0, 1], metrics_by_n_agents_by_gt_by_type[1, 0, 2], metrics_by_n_agents_by_gt_by_type[1, 0, 3],),
    "\\multirow{1}{*}{\\textbf{3 Agents}} & \\cmark & & " + "{:.4f} & {:.4f} & {:.4f} & {:.4f} \\\\".format( metrics_by_n_agents_by_gt_by_type[2, 0, 0], metrics_by_n_agents_by_gt_by_type[2, 0, 1], metrics_by_n_agents_by_gt_by_type[2, 0, 2], metrics_by_n_agents_by_gt_by_type[2, 0, 3],)
]
print("\n".join(table))

\multirow{1}{*}{\textbf{1 Agent}}  & & \cmark & 0.7675 & 0.9130 & 47.4612 & 16.2483 \\
\multirow{1}{*}{\textbf{2 Agents}} & & \cmark & 0.7703 & 0.9710 & 46.5000 & 15.5752 \\
\multirow{1}{*}{\textbf{3 Agents}} & & \cmark & 0.7667 & 1.0000 & 45.6726 & 14.4768 \\
\midrule
\multirow{1}{*}{\textbf{1 Agent}}  & \cmark & & 0.8552 & 0.5072 & 25.6035 & 16.5337 \\
\multirow{1}{*}{\textbf{2 Agents}} & \cmark & & 0.8394 & 0.5290 & 26.6764 & 15.7259 \\
\multirow{1}{*}{\textbf{3 Agents}} & \cmark & & 0.8333 & 0.5435 & 26.7206 & 14.4444 \\


In [23]:
counts_by_n_agents_by_gt_by_type

array([[[9., 9., 9., 9.],
        [9., 9., 9., 9.]],

       [[9., 9., 9., 9.],
        [9., 9., 9., 9.]],

       [[3., 3., 3., 3.],
        [3., 3., 3., 3.]]])